# 3.3 Analyse des Cas Tests

Analyser comment la fusion multimodale résout les confusions des approches unimodales sur les cas tests identifiés en Phase 1.

In [1]:
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

# Charger les données
df = pd.read_csv('data/subset_fashion_dataset/products_final.csv')
ground_truth = pd.read_csv('data/subset_fashion_dataset/ground_truth.csv', sep=';')
ground_truth.columns = ground_truth.columns.str.strip()

print(f"Produits: {len(df)}")
print(f"Ancres: {len(ground_truth)}")

Produits: 400
Ancres: 30


In [2]:
# Identifier les cas tests (visual ≠ semantic)
case_tests = []

for _, row in ground_truth.iterrows():
    anchor_id = row['anchor_id']
    
    visual_ids = []
    semantic_ids = []
    
    for i in range(1, 6):
        vid = row[f'visual_id{i}']
        sid = row[f'semantic_id{i}']
        if pd.notna(vid):
            visual_ids.append(int(vid))
        if pd.notna(sid):
            semantic_ids.append(int(sid))
    
    # Trouver les produits visuels qui ne sont pas sémantiques
    visual_only = set(visual_ids) - set(semantic_ids)
    
    if visual_only:
        anchor_info = df[df['id'] == anchor_id].iloc[0]
        case_tests.append({
            'anchor_id': anchor_id,
            'anchor_name': anchor_info['nom'],
            'anchor_category': anchor_info['categorie'],
            'visual_only': list(visual_only)
        })

df_cases = pd.DataFrame(case_tests)
print(f"Cas tests identifiés: {len(df_cases)}")
print(df_cases.head(10))

Cas tests identifiés: 30
   anchor_id                                        anchor_name  \
0       3314              Guerrilla Men's Warrior Brown T-shirt   
1       4543                   Nike Womens Air Rele Purple Shoe   
2      15270  ADIDAS Originals Women Top Ten low Sleek Pink ...   
3      17993        Puma Women Essential Skinny Grey Track Pant   
4      20610           Baggit Women Debbie Sufiya Beige Handbag   
5      22971            U.S. Polo Assn. Unisex Black Laptop Bag   
6      24532                 Mother Earth Women Off white Kurta   
7      24756                          Spykar Men Green Sweaters   
8      25521                     Levis Men White Innerwear Vest   
9      28064                 Jockey ELANCE Men Black Trunk 1015   

  anchor_category                   visual_only  
0         Topwear          [4861, 26486, 31279]  
1           Shoes                [34832, 15743]  
2           Shoes          [17041, 3795, 36476]  
3      Bottomwear  [48497, 27906, 126

In [3]:
# Charger tous les résultats de top-5
def load_top5(filename):
    df = pd.read_csv(filename)
    df['top5_ids'] = df['top5_ids'].apply(lambda x: eval(x) if isinstance(x, str) else x)
    return df

results = {
    'TF-IDF': load_top5('results/tfidf_top5_anchors.csv'),
    'MiniLM': load_top5('results/embeddings_top5_anchors.csv'),
    'ResNet-50': load_top5('results/resnet_top5_anchors.csv'),
    'ViT': load_top5('results/vit_top5_anchors.csv'),
    'Concat': load_top5('results/strategy1_concatenation_top5.csv'),
    'Moyenne': load_top5('results/strategy2_weighted_top5.csv'),
}

print("Résultats chargés:")
for name, df_res in results.items():
    print(f"  {name}: {len(df_res)} ancres")

Résultats chargés:
  TF-IDF: 30 ancres
  MiniLM: 30 ancres
  ResNet-50: 30 ancres
  ViT: 30 ancres
  Concat: 30 ancres
  Moyenne: 30 ancres


In [4]:
# Créer une planche visuelle pour un produit ancre
def create_visual_panel(anchor_id, methods_results, output_path, df_data, image_folder):
    """Créer une planche visuelle comparant toutes les méthodes"""
    n_methods = len(methods_results)
    fig, axes = plt.subplots(n_methods + 1, 6, figsize=(20, 3 * (n_methods + 1)))
    
    # Produit ancre (première ligne)
    anchor_info = df_data[df_data['id'] == anchor_id].iloc[0]
    anchor_img_path = os.path.join(image_folder, f"{anchor_id}.jpg")
    
    axes[0, 0].imshow(Image.open(anchor_img_path))
    axes[0, 0].set_title(f"ANCRE\n{anchor_info['nom']}", fontsize=10, fontweight='bold')
    axes[0, 0].axis('off')
    
    for i in range(1, 6):
        axes[0, i].axis('off')
    
    # Top-5 pour chaque méthode
    for row_idx, (method_name, df_res) in enumerate(methods_results.items(), 1):
        anchor_row = df_res[df_res['anchor_id'] == anchor_id]
        
        if len(anchor_row) == 0:
            continue
        
        top5_ids = anchor_row.iloc[0]['top5_ids']
        
        # Nom de la méthode
        axes[row_idx, 0].text(0.5, 0.5, method_name, 
                              ha='center', va='center', fontsize=11, fontweight='bold')
        axes[row_idx, 0].axis('off')
        
        # Top-5 images
        for col_idx, prod_id in enumerate(top5_ids[:5], 1):
            if col_idx >= 6:
                break
            
            img_path = os.path.join(image_folder, f"{prod_id}.jpg")
            if os.path.exists(img_path):
                img = Image.open(img_path)
                axes[row_idx, col_idx].imshow(img)
                
                prod_info = df_data[df_data['id'] == prod_id].iloc[0]
                category = prod_info['categorie']
                
                # Colorer si même catégorie que l'ancre
                if category == anchor_info['categorie']:
                    color = 'green'
                else:
                    color = 'red'
                
                axes[row_idx, col_idx].set_title(category, fontsize=8, color=color)
            axes[row_idx, col_idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    
    print(f"Planche sauvegardée: {output_path}")

In [5]:
# Créer les planches pour les premiers cas tests
os.makedirs('results/case_studies', exist_ok=True)
image_folder = 'data/subset_fashion_dataset/images'

# Prendre les 5 premiers cas tests
for idx, row in df_cases.head(5).iterrows():
    anchor_id = row['anchor_id']
    output_path = f"results/case_studies/anchor_{anchor_id}_panel.png"
    
    create_visual_panel(anchor_id, results, output_path, df, image_folder)
    print(f"Ancre {anchor_id} ({row['anchor_name']})")
    print(f"  Catégorie: {row['anchor_category']}")
    print(f"  Visuels uniquement: {row['visual_only']}")
    print()

Planche sauvegardée: results/case_studies/anchor_3314_panel.png
Ancre 3314 (Guerrilla Men's Warrior Brown T-shirt)
  Catégorie: Topwear
  Visuels uniquement: [4861, 26486, 31279]

Planche sauvegardée: results/case_studies/anchor_4543_panel.png
Ancre 4543 (Nike Womens Air Rele Purple Shoe)
  Catégorie: Shoes
  Visuels uniquement: [34832, 15743]

Planche sauvegardée: results/case_studies/anchor_15270_panel.png
Ancre 15270 (ADIDAS Originals Women Top Ten low Sleek Pink Casual Shoes)
  Catégorie: Shoes
  Visuels uniquement: [17041, 3795, 36476]

Planche sauvegardée: results/case_studies/anchor_17993_panel.png
Ancre 17993 (Puma Women Essential Skinny Grey Track Pant)
  Catégorie: Bottomwear
  Visuels uniquement: [48497, 27906, 12677, 26822]

Planche sauvegardée: results/case_studies/anchor_20610_panel.png
Ancre 20610 (Baggit Women Debbie Sufiya Beige Handbag)
  Catégorie: Bags
  Visuels uniquement: [35915, 36069, 43227]



In [6]:
# Afficher les cas de confusion déjà identifiés
print(f"Cas de confusion identifiés: {len(df_cases)}")
print("\n5 premiers cas:")
for idx, row in df_cases.head(5).iterrows():
    print(f"\nAncre {row['anchor_id']} ({row['anchor_name']})")
    print(f"  Catégorie: {row['anchor_category']}")
    print(f"  → Confondu avec (visuel uniquement): {row['visual_only']}")
    
    # Afficher les noms des produits confondus
    for prod_id in row['visual_only']:
        prod_info = df[df['id'] == prod_id].iloc[0]
        print(f"     - {prod_info['nom']} ({prod_info['categorie']})")


Cas de confusion identifiés: 30

5 premiers cas:

Ancre 3314 (Guerrilla Men's Warrior Brown T-shirt)
  Catégorie: Topwear
  → Confondu avec (visuel uniquement): [4861, 26486, 31279]
     - Nike Men Arsenal Yellow Jersey (Topwear)
     - Puma Men Large Logo Ringer Blue T-shirt (Topwear)
     - Puma Men Bonn Graphic Navy Blue T-shirt (Topwear)

Ancre 4543 (Nike Womens Air Rele Purple Shoe)
  Catégorie: Shoes
  → Confondu avec (visuel uniquement): [34832, 15743]
     - ADIDAS Men Sukoi White Sports Shoes (Shoes)
     - Nike Men Downshifter 4 MSL White Sports Shoes (Shoes)

Ancre 15270 (ADIDAS Originals Women Top Ten low Sleek Pink Casual Shoes)
  Catégorie: Shoes
  → Confondu avec (visuel uniquement): [17041, 3795, 36476]
     - Gas Men Skate 002 Casual Shoe (Shoes)
     - Converse Unisex CT  LACE COLOR OX Black Casual Shoes (Shoes)
     - Spinn Men Agile Black Shoes (Shoes)

Ancre 17993 (Puma Women Essential Skinny Grey Track Pant)
  Catégorie: Bottomwear
  → Confondu avec (visuel unique

In [7]:
# Analyse de confusion pour un cas spécifique
def analyze_confusion(anchor_id, methods_results, df_data, ground_truth_df):
    """Analyse comment chaque méthode recommande les produits"""
    anchor_info = df_data[df_data['id'] == anchor_id].iloc[0]
    gt_row = ground_truth_df[ground_truth_df['anchor_id'] == anchor_id].iloc[0]
    
    print(f"=== ANCRE: {anchor_info['nom']} ({anchor_info['categorie']}) ===\n")
    
    for method_name, df_res in methods_results.items():
        anchor_row = df_res[df_res['anchor_id'] == anchor_id]
        
        if len(anchor_row) == 0:
            continue
        
        top5_ids = anchor_row.iloc[0]['top5_ids']
        
        # Compter combien sont de la même catégorie
        same_category = 0
        for prod_id in top5_ids:
            prod_info = df_data[df_data['id'] == prod_id].iloc[0]
            if prod_info['categorie'] == anchor_info['categorie']:
                same_category += 1
        
        print(f"{method_name:12s}: {same_category}/5 même catégorie")

# Analyser les 3 premiers cas tests
for idx, row in df_cases.head(3).iterrows():
    analyze_confusion(row['anchor_id'], results, df, ground_truth)
    print()

=== ANCRE: Guerrilla Men's Warrior Brown T-shirt (Topwear) ===

TF-IDF      : 5/5 même catégorie
MiniLM      : 3/5 même catégorie
ResNet-50   : 5/5 même catégorie
ViT         : 5/5 même catégorie
Concat      : 5/5 même catégorie
Moyenne     : 5/5 même catégorie

=== ANCRE: Nike Womens Air Rele Purple Shoe (Shoes) ===

TF-IDF      : 5/5 même catégorie
MiniLM      : 3/5 même catégorie
ResNet-50   : 5/5 même catégorie
ViT         : 5/5 même catégorie
Concat      : 5/5 même catégorie
Moyenne     : 5/5 même catégorie

=== ANCRE: ADIDAS Originals Women Top Ten low Sleek Pink Casual Shoes (Shoes) ===

TF-IDF      : 4/5 même catégorie
MiniLM      : 2/5 même catégorie
ResNet-50   : 5/5 même catégorie
ViT         : 5/5 même catégorie
Concat      : 5/5 même catégorie
Moyenne     : 5/5 même catégorie



## Conclusions

### Observations:
1. **Confusions résolues**: Quelles méthodes multimodales corrigent les erreurs visuelles/sémantiques ?
2. **Cas problématiques**: Quels produits restent mal recommandés même avec fusion ?
3. **Meilleure approche**: Quelle stratégie de fusion donne les meilleurs résultats sur les cas tests ?

### Planches visuelles générées dans `results/case_studies/`